# CRE — F1 and F8 ONLY, random PMCID sample

**Engine commit `8f28bda330226910c817bd4127d7e2ec31d50714`** on `merge/f2-into-f3f7`
(2026-08-23, "Pay OpenAlex on every leg, and keep a spent allowance reading as
'could not look'"). This is the commit that makes F1 reachable at all — before it,
`confirm.search_openalex` could not authenticate, so `fully_answered` went False for
the whole corpus once the anonymous allowance was spent. The tree also carries
`5602a3b`, the F8 dating fix.

## What this notebook runs, and what it does not

It runs **Band 1 only** — `cre.f1.run.run(..., f8_timing=True)`, the exact function
`production_launcher.launch_full` calls at line 982. It then **stops**. It never
calls `production_launcher.launch(...)`, so **F3, F4, F5, F6 and F7 never execute**.

**One honest caveat you have to know about.** Band 1 is a single pass that decides
F1, F2 and F8 together inside `run.process_reference`. The lookup, the flag, the
LLM filter and the confirmation searches are shared machinery — F1 and F2 come out
of the same `decide()` call. There is no switch that runs F1 and F8 without F2, and
cutting one would mean editing engine code, which the project's no-rewrite
discipline forbids. So this notebook:

- runs the **real, unmodified** Band-1 code;
- keeps every raw artifact, F2 rows included, in the run directory;
- **reports and grades only F1 and F8 rows.** F2 rows are excluded from the
  reported set and are never counted in any numerator or denominator here.

Say it that way in any write-up: *"F2 was computed by the shared Band-1 pass and
excluded from the reported set,"* not *"F2 was not run."*

## Sampling

A **uniform random sample of 1,000 papers** drawn from a near-complete frame: PMC
`"open access"[filter]` for 2024–2025, enumerated one publication **day** at a time
so no single ESearch window truncates the frame. The draw is seeded and the whole
frame is hashed, so the sample is reproducible.

**Section 6 runs 100 papers per execution and stops.** Rerun it for the next 100,
up to the 1,000 cap. Each batch ends with a provider re-probe and prints whether F1
is reportable for that batch, so you decide with evidence between batches instead of
discovering a dead provider ten hours in.

## Cost and time

Measured on 2026-08-24: **$5.18 over 530 model calls for 174 papers / 8,009
references** — but roughly 47% of that was duplicate work from a since-fixed
run-naming defect, so treat it as an upper bound per paper.

Wall clock is the interesting part. Band 1 resolves **one PMID per request** and has
no disk cache, which is why a naive run is 6–10 hours. Section 5C fixes that the way
F2 already does it — batch the unique PMIDs, 200 per request — and the estimate drops
to roughly **45–90 minutes** per shard. Shard across N Colab runtimes to divide that
again. Every paper checkpoints to Drive, so a dropped runtime costs one paper.

**Read the heartbeat, not these numbers.** They come from a request-count model; the
first minute of `refs/s` tells you the truth.

## The F1 reportability gate — read this before trusting any zero

`confirm.fully_answered` requires PubMed, Crossref **and** OpenAlex to all answer
before F1 is reachable. If any cannot answer, F1 is structurally impossible and a
zero says nothing about the literature.

This is not hypothetical. On 2026-08-24 two probes from the same run disagreed —
OpenAlex scored 100.0 early and returned `None` later, while PubMed and Crossref
returned identical scores both times. The anonymous allowance ran out mid-run, and
**F1 = 0 over 8,009 references was an artifact of API access.**

Engine commit `8f28bda` fixed the cause: every OpenAlex leg now takes an API key.
This notebook additionally refuses to trust a single probe. It probes **before** the
batch and **again after**, counts `409`s on the confirm leg from
`openalex_telemetry`, and reports `F1_REPORTABLE` only when all three hold. If it
does not hold, record F1 as **NOT ATTEMPTED** for those papers, never as zero. F8
never calls `confirm()` and is unaffected either way.

## Before running

1. Colab secrets `ANTHROPIC_API_KEY`, `NCBI_API_KEY`, and `OPENALEX_API_KEY`.
2. Run top to bottom. Section 6 stays locked until you set `ENABLE_PAID_RUN = True`.
3. **After any push to the branch, restart the Colab runtime** (Runtime > Restart
   session) before rerunning. `cre/` has no `__init__.py`; modules load by path, and
   `importlib.invalidate_caches()` does not evict what is already in `sys.modules`.

Running F1 and F8 counts print live while the run is active.

## 1. Install, mount Drive, and load the exact engine commit

In [ ]:
# Estimated runtime: 1-3 minutes (pip install, clone, import). Cached clone: ~20 s.
import os, io, sys, re, json, time, gzip, random, shutil, hashlib, threading
import subprocess, collections
import xml.etree.ElementTree as ET
from pathlib import Path
from datetime import datetime, timezone, date, timedelta
from urllib.parse import urlparse
from concurrent.futures import ThreadPoolExecutor, as_completed

# 8f28bda, 2026-08-23 23:08 -0500: "Pay OpenAlex on every leg, and keep a spent
# allowance reading as 'could not look'". This is the commit that makes F1
# reachable at all -- before it, search_openalex could not authenticate, so
# confirm.fully_answered went False for the whole corpus once the anonymous
# $0.10/day allowance was spent. Do NOT run an F1 corpus pass on an older tree.
EXPECTED_COMMIT = "8f28bda330226910c817bd4127d7e2ec31d50714"
BRANCH = "merge/f2-into-f3f7"
REPO_URL = "https://github.com/astonliu/citation-repair-engine.git"
REPO = Path("/content/citation-repair-engine")
PKG_ROOT = REPO / "citation_repair_F1_handoff"

subprocess.run([
    sys.executable, "-m", "pip", "-q", "install",
    "rapidfuzz==3.14.5", "requests==2.32.5", "lxml==6.0.2",
    "anthropic==0.122.0", "jsonschema==4.26.0",
], check=True)

from google.colab import drive, userdata
drive.mount("/content/drive", force_remount=False)

if any(name == "cre" or name.startswith("cre.f1") for name in sys.modules):
    raise RuntimeError("CRE is already imported. Restart the Colab runtime and rerun.")

if REPO.exists():
    if not (REPO / ".git").exists():
        raise RuntimeError(f"{REPO} exists but is not the expected git checkout")
    dirty = subprocess.check_output(
        ["git", "-C", str(REPO), "status", "--porcelain"], text=True).strip()
    if dirty:
        raise RuntimeError(f"Refusing a dirty Colab checkout:\n{dirty}")
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)

# fetch + detach at the exact commit, never a plain `git pull`.
subprocess.run(["git", "-C", str(REPO), "fetch", "origin", BRANCH], check=True)
known = subprocess.run(
    ["git", "-C", str(REPO), "cat-file", "-e", f"{EXPECTED_COMMIT}^{{commit}}"]
).returncode == 0
if not known:
    raise RuntimeError(
        f"origin/{BRANCH} does not contain {EXPECTED_COMMIT}. Push the branch, "
        "restart the runtime, and rerun this cell.")
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", EXPECTED_COMMIT],
               check=True)

HEAD = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
tracked_dirty = subprocess.check_output(
    ["git", "-C", str(REPO), "status", "--porcelain", "--untracked-files=no"],
    text=True).strip()
assert HEAD == EXPECTED_COMMIT and not tracked_dirty

import importlib
importlib.invalidate_caches()
sys.path.insert(0, str(PKG_ROOT))

from cre.f1 import (
    parser, preband_disposition, production_launcher, ratelimit, ncbi_meta,
    openalex_telemetry,
)
from cre.f1 import run as f1_run
from cre.f1 import lookup as f1_lookup
from cre.f1.schema import (RetrievedRecord, FETCH_NOT_ATTEMPTED,
                           FETCH_ANSWERED_RECORD, FETCH_ANSWERED_ABSENT,
                           FETCH_RESOLVER_ERROR)
from cre.f1.f5_candidate_finder import PubMedCandidateFinder
from dataclasses import asdict

# The Band-1-only path, asserted rather than assumed.
assert callable(f1_run.run) and callable(f1_run.process_reference)
# THE F1 GATE, asserted rather than assumed: an unkeyed OpenAlex leg silently
# turns every F1 into a hold.
import inspect as _inspect
assert "api_key" in _inspect.signature(
    __import__("cre.f1.confirm", fromlist=["confirm"]).search_openalex).parameters, \
    "confirm.search_openalex has no api_key parameter -- this tree predates 8f28bda"
assert "openalex_api_key" in _inspect.signature(f1_run.run).parameters, \
    "run.run has no openalex_api_key parameter -- this tree predates 8f28bda"
assert callable(preband_disposition.write_disposition)
assert "f8_retraction.py" in production_launcher.GOVERNING_MODULES
TREE_RECEIPT = production_launcher.verify_tree(str(REPO), str(PKG_ROOT / "cre/f1"))

DRIVE_ROOT = Path("/content/drive/MyDrive/CitationRepairEngine")
CACHE_ROOT = DRIVE_ROOT / "cache"
RUNS_ROOT = DRIVE_ROOT / "runs"
FRAME_ROOT = DRIVE_ROOT / "sampling_frames"
for d in (CACHE_ROOT, RUNS_ROOT, FRAME_ROOT):
    d.mkdir(parents=True, exist_ok=True)

print("ENGINE COMMIT:", HEAD)
print("BAND 1 ONLY  : cre.f1.run.run(..., f8_timing=True); no judgment_run, no F3-F7")
print("GOVERNED MODULES:", len(production_launcher.GOVERNING_MODULES))
print("DRIVE ROOT:", DRIVE_ROOT)
print("READY")

## 2. Run configuration and credentials

In [ ]:
# Estimated runtime: under 5 seconds.
# DEC-084 (2026-08-24, ZD): Band-1 llm_filter runs on Haiku 4.5, not Opus 5.
#   WHY: cost. Opus measured $4.5988 / 327 complete papers ($0.0141 each) on
#   f1f8_only_seed20260824_n1000. A multi-thousand-paper run at that rate is not
#   affordable.
#   MODEL SWAP ONLY. The llm_filter prompt, the decide() conjunction and every
#   other stage are untouched, so nothing here is attributable to a prompt change.
#   SEPARATE STRATA (DEC-083): Opus-run and Haiku-run papers are never pooled in
#   one F1 or F2 denominator. RUN_NAME carries the model for exactly this reason.
#   THE NOVELTY CLAIM IS SPENT: CITADEL used Claude 3.5 Haiku zero-shot, so this
#   stage no longer carries a model-side improvement over it. "A later Haiku,
#   same zero-shot posture" is defensible; "a stronger model" is not.
MODEL = "claude-haiku-4-5"
EMAIL = "aston.hliu@gmail.com"

# --- sampling -------------------------------------------------------------
FRAME_QUERY   = '"open access"[filter]'   # the day filter is appended per day
FRAME_START   = date(2024, 1, 1)
FRAME_END     = date(2025, 12, 31)
SAMPLE_SIZE   = 1000        # the CAP. The whole draw, never exceeded.
BATCH_SIZE    = 100         # papers per execution of Section 6; then it stops.
SAMPLE_SEED   = 20260824

# --- sharding: the only lever that buys real wall-clock ---------------------
# NCBI's rate limit is per SITE (IP), not per key, and NCBI allows one key per
# account -- so a second key in this runtime buys nothing. A second RUNTIME has
# its own IP and its own 9 req/s. Open N Colab runtimes, set SHARD_COUNT = N and
# SHARD_INDEX = 0..N-1, run the same notebook in each. Papers are disjoint by
# construction, they write into per-paper directories under the SAME RUN_NAME on
# Drive, and Section 7 aggregates whatever is there.
# Set RUN_NAME by hand to the SAME string in every shard before running.
SHARD_COUNT   = 1
SHARD_INDEX   = 0

# --- concurrency ----------------------------------------------------------
# Papers run in parallel; NCBI politeness is enforced globally by the engine's own
# shared limiter (ratelimit.NCBI, 9 req/s with a key), which every Band-1 network
# call already goes through. Raising this does not raise the NCBI rate; it only
# keeps the limiter saturated and overlaps the model calls.
# 24, not 12: once the two NCBI legs are prewarmed in batches (Section 5C) the
# per-reference NCBI cost collapses and the shared 9 req/s limiter stops being
# the floor. Residual traffic is Crossref/OpenAlex on flagged survivors plus the
# model calls, and those have their own limiters.
MAX_WORKERS = 24
MAX_INFLIGHT_MODEL_CALLS = MAX_WORKERS

# --- provider rates ---------------------------------------------------------
# ratelimit.py ships the ANONYMOUS-pool numbers. With a mailto we are in the
# polite pools, which are documented higher. request_with_retry backs off on 429
# and (at this commit) caps any Retry-After at max_backoff, so a raised rate
# degrades into backoff rather than a stranded worker.
NCBI_RATE     = 9.0     # 10/s documented with a key; 9 leaves headroom
CROSSREF_RATE = 10.0    # Crossref polite pool = 10 per interval (public = 5)
OPENALEX_RATE = 5.0     # left alone: OpenAlex is credit-limited, not rate-limited

# --- batched prewarm: the single biggest speed lever -------------------------
# Band 1 resolves ONE PMID PER REQUEST and has no disk cache. F2's seed-47 run
# scored 57,459 citation occurrences in minutes because it batched EFetch over
# unique PMIDs and reused a 54,500-record cache. This reproduces that shape at
# the notebook level: batch the two NCBI legs up front, then let Band 1 read
# them. PREWARM_VERIFY_N records are re-fetched ONE AT A TIME and compared
# field-for-field against the batched result; any mismatch aborts the run.
PREWARM_PUBMED   = True
PREWARM_F8       = True
EFETCH_BATCH     = 200        # NCBI's documented ceiling for a GET id list
PREWARM_VERIFY_N = 40         # half drawn from records that came back ABSENT
MODEL_MAX_TOKENS = 400            # matches run.make_completer's default
ANTHROPIC_MAX_RETRIES = 3

# --- the gate -------------------------------------------------------------
ENABLE_PAID_RUN = False           # read Section 6, then set True

# --- reference-count guard ------------------------------------------------
# Same window the 2026-08-23 overnight run used. A 3-reference editorial and a
# 900-reference review are both bad value per NCBI second.
MIN_REFERENCES, MAX_REFERENCES = 8, 200

HEARTBEAT_SECONDS = 30

# DETERMINISTIC, not a timestamp. A timestamped RUN_NAME is regenerated every
# time a runtime restarts, which silently forks the work into a second directory
# that the resume scan cannot see -- it cost 297 completed papers on 2026-08-24.
# Same seed + same sample size + same commit = same run root, so every restart
# and every shard resumes into one place. Change SAMPLE_SEED for a new run.
# THE MODEL IS PART OF THE RUN IDENTITY. Without it a Haiku run resumes into the
# Opus root, skips the papers already done there, and puts two strata in one
# directory -- which DEC-083 and DEC-084 consequence 1 both forbid. Same seed,
# same draw, different model = different run root.
MODEL_TAG = MODEL.replace("claude-", "").replace("-", "")      # opus5 | haiku45
RUN_NAME = "f1f8_only_seed%d_n%d_%s" % (SAMPLE_SEED, SAMPLE_SIZE, MODEL_TAG)
SNAPSHOT_DATE = datetime.now(timezone.utc).date().isoformat()

def secret(name):
    try:
        value = userdata.get(name) or ""
    except Exception:
        value = ""
    if not value:
        raise RuntimeError(f"Add {name} under Colab > Secrets, then rerun this cell")
    return value

ANTHROPIC_API_KEY = secret("ANTHROPIC_API_KEY")
NCBI_API_KEY = secret("NCBI_API_KEY")
OPENALEX_API_KEY = secret("OPENALEX_API_KEY")   # required; F1 is a hold without it
ratelimit.configure_ncbi(True)
ratelimit.NCBI.set_rate(NCBI_RATE)
ratelimit.CROSSREF.set_rate(CROSSREF_RATE)
ratelimit.OPENALEX.set_rate(OPENALEX_RATE)

# Claude Haiku 4.5 public API prices, USD per million tokens, checked 2026-08-24.
# Base $1.00 in / $5.00 out; the two cache rates are the documented 1.25x write /
# 0.10x read multipliers -- the same derivation the Opus row used (5.00 -> 6.25 /
# 0.50). EVERY spend_usd FIGURE PRINTED BEFORE DEC-084 IS AN OPUS FIGURE and must
# be labelled as one; do not compare them without saying which model produced them.
PRICE_USD_PER_MTOK = {
    "input_tokens": 1.00,
    "cache_creation_input_tokens": 1.25,
    "cache_read_input_tokens": 0.10,
    "output_tokens": 5.00,
}
PRICE_SOURCE = "https://platform.claude.com/docs/en/about-claude/pricing"
PRICE_CHECKED_DATE = "2026-08-24"

# ---- FAST LOCAL MIRROR -----------------------------------------------------
# Google Drive is mounted over FUSE. Every stat, open, write and fsync is a
# network round trip, and it collapses under many small operations from many
# threads. The project already knows this -- MASS_RUN_NOTEBOOK_2026-08-22 item 8
# is exactly this fix ("Fast local mirror: caches and run outputs on
# /content/cre_fast, rsynced to Drive after every batch").
#
# So: every HOT path lives on local disk and is rsynced to Drive at checkpoints.
# Drive stays the durable home; it is never in the inner loop.
FAST_ROOT   = Path("/content/cre_fast")
FAST_RUN    = FAST_ROOT / RUN_NAME
PAPERS_ROOT = FAST_RUN / "papers"          # hot
EVENT_ROOT  = FAST_RUN / "events"          # hot
XML_CACHE   = FAST_ROOT / "cache" / "citing_xml"   # hot
MEASURE_ROOT= FAST_RUN / "measurements"
REVIEW_ROOT = FAST_RUN / "review"

RUN_ROOT      = RUNS_ROOT / RUN_NAME       # durable, Drive
DRIVE_XML     = CACHE_ROOT / "citing_xml"  # durable, Drive
for d in (PAPERS_ROOT, EVENT_ROOT, MEASURE_ROOT, REVIEW_ROOT, XML_CACHE,
          RUN_ROOT, DRIVE_XML):
    d.mkdir(parents=True, exist_ok=True)

FINDINGS_LIVE = FAST_RUN / "f1_f8_findings_live.jsonl"
FAILURES      = FAST_RUN / "paper_failures.jsonl"

def paper_dir(pmcid):
    return PAPERS_ROOT / pmcid

_LAST_SYNC = [0.0]
SYNC_EVERY_SECONDS = 300        # cap how much a runtime death can cost

def sync_to_drive(label="", min_interval=0.0):
    """Push the fast mirror to Drive. Called at checkpoints, never in a loop.

    `min_interval` makes a call a no-op if the last sync was recent, so the run
    loop can ask on every paper and still only pay every few minutes."""
    if min_interval and (time.time() - _LAST_SYNC[0]) < min_interval:
        return
    _LAST_SYNC[0] = time.time()
    t0 = time.time()
    for src, dst in ((FAST_RUN, RUN_ROOT), (XML_CACHE, DRIVE_XML)):
        subprocess.run(["rsync", "-a", "--no-perms", "--no-owner", "--no-group",
                        str(src) + "/", str(dst) + "/"], check=False)
    print("[sync] %s -> Drive in %.0f s" % (label or "checkpoint",
                                            time.time() - t0), flush=True)

def seed_from_drive():
    """ONE bulk pull of the durable state into the fast mirror, at start.

    This is also what makes RESUME work after a runtime drop: the per-paper
    status.json files live on Drive between sessions and come back here in one
    rsync instead of 2,000 FUSE stats."""
    t0 = time.time()
    for src, dst in ((DRIVE_XML, XML_CACHE), (RUN_ROOT, FAST_RUN)):
        subprocess.run(["rsync", "-a", "--no-perms", "--no-owner", "--no-group",
                        str(src) + "/", str(dst) + "/"], check=False)
    n_xml = sum(1 for _ in XML_CACHE.glob("*.xml"))
    n_done = sum(1 for _ in PAPERS_ROOT.glob("*/status.json"))
    print("[seed] %d cached XML, %d finished papers pulled from Drive in %.0f s"
          % (n_xml, n_done, time.time() - t0), flush=True)

print("RUN:", RUN_NAME, "| model:", MODEL)
print("SAMPLE:", SAMPLE_SIZE, "papers, seed", SAMPLE_SEED,
      "| frame", FRAME_START, "to", FRAME_END)
print("WORKERS:", MAX_WORKERS,
      "| rates NCBI %.0f/s, Crossref %.0f/s, OpenAlex %.0f/s"
      % (NCBI_RATE, CROSSREF_RATE, OPENALEX_RATE))
print("PREWARM: pubmed", PREWARM_PUBMED, "| f8", PREWARM_F8,
      "| batch", EFETCH_BATCH, "| verify", PREWARM_VERIFY_N)
print("BATCHING: %d papers per run of Section 6, cap %d" % (BATCH_SIZE, SAMPLE_SIZE))
print("Secrets found: ANTHROPIC_API_KEY, NCBI_API_KEY, OPENALEX_API_KEY (hidden)")
print("REPORTED TAXONOMIES: F1, F8 only  (F2 computed by the same pass, excluded)")
print("FAST MIRROR:", FAST_RUN, "(hot)   DRIVE:", RUN_ROOT, "(durable)")
print("PAID RUN ENABLED:", ENABLE_PAID_RUN)

## 3. Durable telemetry, HTTP logging, and the Band-1 model transport

In [ ]:
# Estimated runtime: under 5 seconds.
import requests
from anthropic import Anthropic

_LOCKS_GUARD = threading.Lock()
_LOCKS = {}

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def file_lock(path):
    key = str(Path(path))
    with _LOCKS_GUARD:
        return _LOCKS.setdefault(key, threading.Lock())

def append_jsonl(path, record):
    """Append one JSON line. NO fsync: these files live on the fast local mirror
    and are rsynced to Drive at checkpoints. An fsync per HTTP event on a FUSE
    mount serialises every worker behind a network round trip -- that is what
    made the first 5C attempt print nothing for 18 minutes."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with file_lock(path), path.open("a", encoding="utf-8") as fh:
        fh.write(json.dumps(record, sort_keys=True, ensure_ascii=False) + "\n")

def read_jsonl(path):
    path = Path(path)
    if not path.exists():
        return []
    return [json.loads(x) for x in
            path.read_text(encoding="utf-8").splitlines() if x.strip()]

def atomic_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(payload, indent=2, sort_keys=True,
                              ensure_ascii=False) + "\n", encoding="utf-8")
    os.replace(tmp, path)

HTTP_EVENTS  = EVENT_ROOT / "http_events.jsonl"
MODEL_EVENTS = EVENT_ROOT / "model_events.jsonl"

class LoggedThreadLocalSession:
    """One requests.Session per worker, with redacted durable telemetry."""
    def __init__(self, stage):
        self.stage = stage
        self.local = threading.local()
    def _client(self):
        if not hasattr(self.local, "client"):
            self.local.client = requests.Session()
        return self.local.client
    def request(self, method, url, **kwargs):
        started = time.perf_counter()
        safe = {str(k): str(v)[:300] for k, v in dict(kwargs.get("params") or {}).items()
                if str(k).lower() not in {"api_key", "key", "token"}}
        event = {"ts": utc_now(), "stage": self.stage, "method": method,
                 "host": urlparse(url).netloc, "path": urlparse(url).path,
                 "params": safe}
        try:
            r = self._client().request(method, url, **kwargs)
            event.update(result="response", status_code=r.status_code,
                         elapsed_s=round(time.perf_counter() - started, 4),
                         response_bytes=len(r.content or b""))
            return r
        except Exception as exc:
            event.update(result="exception", exception_type=type(exc).__name__,
                         message=str(exc)[:500],
                         elapsed_s=round(time.perf_counter() - started, 4))
            raise
        finally:
            append_jsonl(HTTP_EVENTS, event)
    def get(self, url, **kw):  return self.request("GET", url, **kw)
    def post(self, url, **kw): return self.request("POST", url, **kw)

SESSION = LoggedThreadLocalSession("frame_and_corpus")

# ---- thread-safe stdout routing -------------------------------------------
# Band 1 prints an eval_report and its quarantine lines on every call. With 12
# worker threads, contextlib.redirect_stdout would NOT work: it swaps the single
# global sys.stdout, so one worker's capture would swallow another worker's F1/F8
# line. This router is installed once and keys on the CALLING THREAD, so a worker
# that has parked a buffer captures only its own output and everything else --
# the findings lines and the heartbeat -- still reaches the console.
class _ThreadRoutedStdout:
    def __init__(self, real):
        self.real = real
        self.local = threading.local()
    def park(self, buffer):
        self.local.buffer = buffer
    def release(self):
        self.local.buffer = None
    def _target(self):
        return getattr(self.local, "buffer", None) or self.real
    def write(self, data):
        return self._target().write(data)
    def flush(self):
        try:
            return self._target().flush()
        except Exception:
            return None
    def isatty(self):
        return False

if not isinstance(sys.stdout, _ThreadRoutedStdout):
    sys.stdout = _ThreadRoutedStdout(sys.stdout)
STDOUT_ROUTER = sys.stdout

# ---- Band-1 model transport ------------------------------------------------
# Band 1's `complete` seam is a plain str -> str callable (run.make_completer).
# No prompt caching here: Band-1 prompts are short, per-reference and share no
# prefix, so a breakpoint would buy a 1.25x write and no read.
_anthropic_local = threading.local()
def anthropic_client():
    if not hasattr(_anthropic_local, "client"):
        _anthropic_local.client = Anthropic(
            api_key=ANTHROPIC_API_KEY, max_retries=ANTHROPIC_MAX_RETRIES,
            timeout=180.0)
    return _anthropic_local.client

MODEL_INFLIGHT = threading.BoundedSemaphore(MAX_INFLIGHT_MODEL_CALLS)

class Spend:
    """Thread-safe running token/cost tally, read by the heartbeat."""
    def __init__(self):
        self.lock = threading.Lock()
        self.calls = 0
        self.tokens = {k: 0 for k in PRICE_USD_PER_MTOK}
    def add(self, usage):
        with self.lock:
            self.calls += 1
            for k in self.tokens:
                self.tokens[k] += int(usage.get(k, 0) or 0)
    def usd(self):
        with self.lock:
            return sum(self.tokens[k] / 1e6 * v
                       for k, v in PRICE_USD_PER_MTOK.items())

SPEND = Spend()

def make_band1_transport():
    def transport(prompt):
        started = time.perf_counter()
        event = {"ts": utc_now(), "stage": "band1", "model": MODEL,
                 "prompt_sha256": sha256_bytes(prompt.encode("utf-8")),
                 "prompt_chars": len(prompt), "max_tokens": MODEL_MAX_TOKENS}
        try:
            with MODEL_INFLIGHT:
                response = anthropic_client().messages.create(
                    model=MODEL, max_tokens=MODEL_MAX_TOKENS,
                    messages=[{"role": "user", "content": prompt}])
            text = "".join(b.text for b in response.content
                           if getattr(b, "type", None) == "text")
            usage = getattr(response, "usage", None)
            usage_d = {k: int(getattr(usage, k, 0) or 0) for k in PRICE_USD_PER_MTOK}
            SPEND.add(usage_d)
            event.update(result="success", output_chars=len(text),
                         elapsed_s=round(time.perf_counter() - started, 4),
                         **usage_d)
            return text
        except Exception as exc:
            event.update(result="exception", exception_type=type(exc).__name__,
                         status_code=getattr(exc, "status_code", None),
                         message=str(exc)[:500],
                         elapsed_s=round(time.perf_counter() - started, 4))
            raise
        finally:
            append_jsonl(MODEL_EVENTS, event)
    transport.model_id = MODEL
    transport.thread_safe = True
    return transport

BAND1_COMPLETE = make_band1_transport()
print("Telemetry root:", EVENT_ROOT, "(local; synced to Drive at checkpoints)")
print("Band-1 transport ready | model:", MODEL, "| max_tokens:", MODEL_MAX_TOKENS)
print("Max in-flight model calls:", MAX_INFLIGHT_MODEL_CALLS)

## 3A. Model check — one live call, before anything paid

DEC-084's first acceptance row. A wrong model identifier must fail here, loudly,
not silently somewhere inside a paid batch.

In [ ]:
# Estimated runtime: ~10-20 seconds. ONE live model call, a few hundred tokens.
print("MODEL            :", MODEL)
print("transport reports:", BAND1_COMPLETE.model_id)
print("max_tokens       :", MODEL_MAX_TOKENS)
print("prices $/MTok    :", PRICE_USD_PER_MTOK)
print("checked          :", PRICE_CHECKED_DATE, "|", PRICE_SOURCE)

reply = BAND1_COMPLETE("Reply with the single word: ok")
print("live reply       :", repr(reply.strip()[:80]))
assert reply.strip(), ("empty reply -- the model string is probably wrong. A bad "
                       "identifier raises not_found_error rather than returning "
                       "nothing, so check the traceback above too.")
print("\nLIVE 200 CONFIRMED for", MODEL)
print("\nNEXT, AND IT IS THE GATE: run tools/F1_CALIBRATION_PROBE.py under this")
print("model. Probe A -- dead PMID, claimed work absent from all three databases")
print("-- MUST come back F1. If a weaker model calls it formatting_discrepancy,")
print("decide() clears it, F1 stops firing, and every F1 count below is a zero")
print("that means nothing. Probe B must stay cleared.")

## 4. Build the sampling frame, then draw a uniform random sample

**Why day-by-day.** A single PMC ESearch returns at most ~10,000 ids and orders
them by recency, so `retmax=10000` over 2024–2025 is not a random slice of the
~1.7 M hits — it is the newest 10,000. Enumerating one publication **day** at a
time keeps each window under the cap, so the union is a near-complete frame and a
seeded `random.sample` over it is a genuine uniform draw.

Any day that still exceeds the cap is recorded in the frame manifest as
`truncated_days`. Report that number; it is the only place the frame is
incomplete.

The frame is cached on Drive and hashed, so the same seed reproduces the same
2,000 PMCIDs without re-querying NCBI.

In [ ]:
# Estimated runtime: ~5-12 minutes on the first build (731 ESearch calls at 9 req/s
# plus response time, ~1.7M ids downloaded). Under 30 seconds from the Drive cache.
FRAME_TAG = "pmc_oa_%s_%s" % (FRAME_START.isoformat(), FRAME_END.isoformat())
FRAME_PATH = FRAME_ROOT / (FRAME_TAG + ".jsonl.gz")
FRAME_MANIFEST = FRAME_ROOT / (FRAME_TAG + "_manifest.json")
ESEARCH_RETMAX = 9999

def normalize_pmcid(value):
    value = str(value).strip().upper()
    if value.isdigit():
        value = "PMC" + value
    if not re.fullmatch(r"PMC\d+", value):
        raise ValueError(f"Invalid PMCID: {value!r}")
    return value

def esearch_day(day):
    """(ids, total_count) for one publication date. Truncation is reported, not hidden."""
    term = '%s AND %s[pdat]' % (FRAME_QUERY, day.strftime("%Y/%m/%d"))
    ratelimit.NCBI.wait()
    r = SESSION.get("https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi",
                    params={"db": "pmc", "term": term, "retmax": ESEARCH_RETMAX,
                            "retmode": "json", "tool": "CitationRepairEngine",
                            "email": EMAIL, "api_key": NCBI_API_KEY}, timeout=90)
    r.raise_for_status()
    res = r.json()["esearchresult"]
    return [normalize_pmcid(x) for x in res.get("idlist", [])], int(res.get("count", 0))

if FRAME_PATH.exists() and FRAME_MANIFEST.exists():
    frame_manifest = json.loads(FRAME_MANIFEST.read_text(encoding="utf-8"))
    print("[frame] reusing cached frame:", FRAME_PATH.name)
else:
    days = [FRAME_START + timedelta(days=i)
            for i in range((FRAME_END - FRAME_START).days + 1)]
    rows, truncated, started = [], [], time.time()
    for i, day in enumerate(days, 1):
        ids, count = esearch_day(day)
        if count > ESEARCH_RETMAX:
            truncated.append({"day": day.isoformat(), "count": count,
                              "returned": len(ids)})
        rows.append({"day": day.isoformat(), "count": count, "ids": ids})
        if i == 1 or i % 25 == 0 or i == len(days):
            got = sum(len(x["ids"]) for x in rows)
            print("[frame] %d/%d days | %d ids | %.0f s elapsed"
                  % (i, len(days), got, time.time() - started), flush=True)
    tmp = FRAME_PATH.with_suffix(".tmp.gz")
    with gzip.open(tmp, "wt", encoding="utf-8") as fh:
        for row in rows:
            fh.write(json.dumps(row, sort_keys=True) + "\n")
    os.replace(tmp, FRAME_PATH)
    frame_manifest = {
        "schema": "cre_pmc_sampling_frame_v1",
        "query": FRAME_QUERY, "start": FRAME_START.isoformat(),
        "end": FRAME_END.isoformat(), "days": len(days),
        "esearch_retmax": ESEARCH_RETMAX,
        "ids_total": sum(len(x["ids"]) for x in rows),
        "ids_unique": len({p for x in rows for p in x["ids"]}),
        "hits_reported_total": sum(x["count"] for x in rows),
        "truncated_days": truncated,
        "truncated_day_count": len(truncated),
        "built_at": utc_now(), "frame_path": str(FRAME_PATH),
    }
    frame_manifest["frame_sha256"] = sha256_file(FRAME_PATH)
    atomic_json(FRAME_MANIFEST, frame_manifest)

with gzip.open(FRAME_PATH, "rt", encoding="utf-8") as fh:
    FRAME = sorted({p for line in fh for p in json.loads(line)["ids"]})

if len(FRAME) < SAMPLE_SIZE:
    raise RuntimeError("frame has %d ids, fewer than SAMPLE_SIZE=%d"
                       % (len(FRAME), SAMPLE_SIZE))

rng = random.Random(SAMPLE_SEED)
FULL_SAMPLE = sorted(rng.sample(FRAME, SAMPLE_SIZE))

# The shard slice. Deterministic and disjoint: paper i goes to shard i % N.
if not (0 <= SHARD_INDEX < SHARD_COUNT):
    raise ValueError("SHARD_INDEX must be in [0, SHARD_COUNT)")
SAMPLE = [p for i, p in enumerate(FULL_SAMPLE) if i % SHARD_COUNT == SHARD_INDEX]

SELECTION = {
    "schema": "cre_f1f8_only_selection_v1",
    "engine_commit": HEAD,
    "reported_taxonomies": ["F1", "F8"],
    "excluded_from_report": ["F2"],
    "exclusion_note": ("Band 1 decides F1, F2 and F8 in one shared pass "
                       "(run.process_reference). F2 was COMPUTED and is retained "
                       "in the raw artifacts; it is excluded from the reported "
                       "set. F3-F7 were never executed."),
    "frame": frame_manifest,
    "frame_ids_unique": len(FRAME),
    "sample_size": SAMPLE_SIZE, "seed": SAMPLE_SEED,
    "sampler": "random.Random(seed).sample over the sorted unique frame",
    "sample_sha256": sha256_bytes("\n".join(FULL_SAMPLE).encode("utf-8")),
    "shard_count": SHARD_COUNT, "shard_index": SHARD_INDEX,
    "shard_size": len(SAMPLE),
    "shard_rule": "index i of the sorted full sample goes to shard i % SHARD_COUNT",
    "selected_at": utc_now(),
    "pmcids": FULL_SAMPLE,
}
atomic_json(FAST_RUN / ("selection_shard%d_of_%d.json" % (SHARD_INDEX, SHARD_COUNT)),
            SELECTION)
atomic_json(RUN_ROOT / ("selection_shard%d_of_%d.json" % (SHARD_INDEX, SHARD_COUNT)),
            SELECTION)

print("FRAME unique PMCIDs :", len(FRAME))
print("FRAME truncated days:", frame_manifest["truncated_day_count"],
      "(days where the true count exceeded the ESearch cap)")
print("FULL SAMPLE         :", len(FULL_SAMPLE), "PMCIDs, seed", SAMPLE_SEED)
print("FULL SAMPLE sha256  :", SELECTION["sample_sha256"][:16])
print("THIS SHARD          : %d of %d -> %d papers"
      % (SHARD_INDEX, SHARD_COUNT, len(SAMPLE)))
print("first five          :", SAMPLE[:5])

## 5A. Helpers and the shared stores

Signature checks, the citing-XML retriever, and the two shared PMID stores that
Section 5C fills in bulk. Nothing here makes a paid call.

In [ ]:
# Estimated runtime: under 5 seconds.
import inspect
sig = inspect.signature(f1_run.run)
for required in ("refs", "complete", "f8_timing", "f8_fetch_meta", "f8_resolve_doi"):
    assert required in sig.parameters, f"run.run has no {required} parameter"
assert "source_pmcid" in inspect.signature(parser.parse_pmc_xml).parameters

# ---- shared stores, filled in bulk by Section 5C ---------------------------
# MEDLINE_STORE : pmid -> raw MEDLINE record text, or "" for VERIFIED ABSENT.
#                 "" is the one value that is evidence, so it is written only
#                 when a batch returned HTTP 200 and simply did not contain that
#                 pmid -- exactly what a single-pmid EFetch signals with an empty
#                 body (lookup.fetch_pubmed: "Answered, and there is no such
#                 record. The one case that is evidence.").
# F8_STORE      : pmid -> record dict, or None for VERIFIED ABSENT.
# A pmid ABSENT FROM A STORE is not an absence -- it falls through to the live
# per-pmid call. An outage must never become a cache entry.
MEDLINE_STORE, F8_STORE = {}, {}
_STORE_LOCK = threading.Lock()
PREWARM_STATS = collections.Counter()

_F8_FINDER = PubMedCandidateFinder(api_key=NCBI_API_KEY, email=EMAIL)
_ORIG_FETCH_PUBMED = f1_run.fetch_pubmed
_f8_doi_mem = {}

def _record_from_chunk(pmid, chunk):
    """Rebuild exactly what lookup.fetch_pubmed returns, using ITS OWN parser."""
    if not pmid:
        return RetrievedRecord(resolved=False,
                               transport_status=FETCH_NOT_ATTEMPTED)
    if chunk == "":
        return RetrievedRecord(resolved=False, pmid=pmid,
                               transport_status=FETCH_ANSWERED_ABSENT)
    rec = f1_lookup._parse_medline(chunk, pmid)
    rec.transport_status = (FETCH_ANSWERED_RECORD if rec.resolved
                            else FETCH_RESOLVER_ERROR)
    return rec

def cached_fetch_pubmed(pmid, api_key="", email="", session=None):
    """Drop-in for lookup.fetch_pubmed. Cache HIT or live delegate, never a guess."""
    key = str(pmid or "").strip()
    if key:
        with _STORE_LOCK:
            chunk = MEDLINE_STORE.get(key, None)
        if chunk is not None:
            PREWARM_STATS["pubmed_hit"] += 1
            return _record_from_chunk(key, chunk)
    PREWARM_STATS["pubmed_live"] += 1
    return _ORIG_FETCH_PUBMED(pmid, api_key, email, session)

def f8_fetch_meta(work_id):
    key = str(work_id or "").strip()
    if not key:
        return None
    with _STORE_LOCK:
        hit = key in F8_STORE
        value = F8_STORE.get(key)
    if hit:
        PREWARM_STATS["f8_hit"] += 1
        return dict(value) if isinstance(value, dict) else None
    PREWARM_STATS["f8_live"] += 1
    value = _F8_FINDER.fetch_metadata(key)   # may raise; NOT cached on failure
    with _STORE_LOCK:
        F8_STORE[key] = value
    return dict(value) if isinstance(value, dict) else None

def f8_resolve_doi(doi):
    key = str(doi or "").strip().casefold()
    if not key:
        return ""
    with _STORE_LOCK:
        if key in _f8_doi_mem:
            return _f8_doi_mem[key]
    value = ncbi_meta.ncbi_doi_to_pmid(key, api_key=NCBI_API_KEY, email=EMAIL)
    with _STORE_LOCK:
        _f8_doi_mem[key] = value
    return value

# ---- citing XML retrieval --------------------------------------------------
def validate_jats(payload):
    if len(payload) < 500:
        raise ValueError(f"implausibly small XML ({len(payload)} bytes)")
    root = ET.fromstring(payload)
    tags = [n.tag.rsplit("}", 1)[-1] for n in root.iter() if isinstance(n.tag, str)]
    if "article" not in tags:
        raise ValueError("no JATS article element")

def fetch_citing_xml(pmcid):
    path = XML_CACHE / f"{pmcid}.xml"
    if path.exists():
        validate_jats(path.read_bytes())
        return path, "cache"
    ratelimit.NCBI.wait()
    r = SESSION.get("https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi",
                    params={"db": "pmc", "id": pmcid[3:], "retmode": "xml",
                            "tool": "CitationRepairEngine", "email": EMAIL,
                            "api_key": NCBI_API_KEY}, timeout=120)
    r.raise_for_status()
    validate_jats(r.content)
    tmp = path.with_suffix(".tmp")
    tmp.write_bytes(r.content)
    os.replace(tmp, path)
    return path, "live"

print("Helpers ready | MEDLINE_STORE and F8_STORE fill in Section 5C")

## 5B. Provider reachability probe — the F1 gate

`confirm.fully_answered` requires **all three** of PubMed, Crossref and OpenAlex to
return a real score before F1 is reachable. Its own docstring: *"An accusation of
fabrication asserts that the work is in no database; that assertion requires every
database to have actually been consulted."* A provider that cannot answer does not
lower F1 — it makes F1 **structurally impossible**, and a zero-F1 result would then
be a statement about API access, not about the literature.

So probe all three before spending anything. F8 is unaffected either way: it is
decided from PubMed publication types and never calls `confirm()`.

In [ ]:
# Estimated runtime: under 10 seconds (three live title searches).
from cre.f1 import confirm as f1_confirm

PROBE_TITLE = "The Hallmarks of Cancer"
probe_hits = {
    "pubmed":   f1_confirm.search_pubmed(PROBE_TITLE, NCBI_API_KEY),
    "crossref": f1_confirm.search_crossref(PROBE_TITLE, EMAIL),
    "openalex": f1_confirm.search_openalex(PROBE_TITLE, EMAIL,
                                           api_key=OPENALEX_API_KEY),
}
F1_REACHABLE = f1_confirm.fully_answered(probe_hits)
atomic_json(MEASURE_ROOT / "provider_probe.json", {
    "schema": "cre_provider_probe_v1", "probe_title": PROBE_TITLE,
    "hits": probe_hits, "f1_reachable": F1_REACHABLE,
    "when": "before_run", "openalex_keyed": bool(OPENALEX_API_KEY),
    "engine_commit": HEAD,
    "note": ("None means the provider did not ANSWER. fully_answered() must be "
             "True or confirm() can never license an F1, whatever the corpus "
             "contains."),
    "ts": utc_now()})

for name, value in probe_hits.items():
    print("  %-9s %s" % (name, "NO ANSWER (None)" if value is None
                         else "answered, score %.1f" % value))
if F1_REACHABLE:
    print("\nF1 REACHABLE - all three providers answered.")
else:
    dead = ", ".join(sorted(k for k, v in probe_hits.items() if v is None))
    print("\n" + "!" * 74)
    print("F1 IS STRUCTURALLY UNREACHABLE. No answer from: %s" % dead)
    print("confirm.fully_answered() will be False on every reference, so decide()")
    print("can never license an F1 no matter what the corpus contains. A zero-F1")
    print("result from this run would be an artifact of provider access, NOT a")
    print("finding about the literature. Record it that way or fix access first.")
    print("F8 is unaffected: it never calls confirm().")
    print("!" * 74)

## 5C. Batched prewarm — the big speed lever

Band 1 resolves **one PMID per request** and has no disk cache. F2's seed-47 run
scored 57,459 citation occurrences in minutes because it batched EFetch over unique
PMIDs and reused a 54,500-record resolution cache. This cell reproduces that shape:

1. fetch and parse every paper in this shard (parsing is local and fast — seed 47
   parsed 1,750 XML in 15 s);
2. collect the unique claimed PMIDs;
3. fetch them in batches of 200 — two legs, the MEDLINE record Band 1 compares
   against and the PubMed metadata F8 dates against;
4. **verify**: re-fetch a sample one at a time through the untouched
   `lookup.fetch_pubmed` and compare the resulting `RetrievedRecord` field for
   field. Any mismatch aborts before a paid call is made.

**This installs a runtime patch on `run.fetch_pubmed`, and that has to be disclosed
rather than buried.** It substitutes a cache read for a network call and nothing
else: the record is parsed by the engine's own `_parse_medline`, an outage is never
written to the store, and a PMID that is not in the store falls through to the
original function. `""` — verified absent — is the only value that is evidence, and
it is written only when a batch answered HTTP 200 without that record, which is
exactly what a single-PMID EFetch signals with an empty body.

The permanent fix is a batched-resolution seam inside `run.py`, which is a repo
change and a Claude Code spec, not a notebook.

In [ ]:
# Estimated runtime: ~8-20 minutes for a 2,000-paper shard, dominated by fetching
# 2,000 citing XML at 9 req/s (~4 min) plus ~200-400 batched EFetch requests
# (~1 min). Cached XML and a warm store make a rerun ~1 minute.
print("[5C] START", utc_now(), flush=True)

# A WATCHDOG, because a cell that prints nothing is indistinguishable from a hang
# and that is exactly how the first attempt failed. This thread prints every 15 s
# no matter what the workers are doing.
_5C_STOP = threading.Event()
_5C_MARK = {"stage": "starting", "n": 0, "total": 0}

def _5c_watchdog():
    t0 = time.time()
    while not _5C_STOP.wait(15):
        if time.time() - t0 > 7200:      # never outlive the cell by hours
            return
        print("[5C..] %s %d/%d | %.0f s elapsed"
              % (_5C_MARK["stage"], _5C_MARK["n"], _5C_MARK["total"],
                 time.time() - t0), flush=True)

_5C_WATCH = threading.Thread(target=_5c_watchdog, daemon=True)
_5C_WATCH.start()

PMID_RE = re.compile(r"^\d+$")
EFETCH_URL = f1_lookup.EFETCH
MEDLINE_PATH = FAST_ROOT / "cache" / "medline_store.jsonl.gz"
MEDLINE_DRIVE = CACHE_ROOT / "medline_store.jsonl.gz"
MEDLINE_PATH.parent.mkdir(parents=True, exist_ok=True)
if MEDLINE_DRIVE.exists() and not MEDLINE_PATH.exists():
    shutil.copy2(MEDLINE_DRIVE, MEDLINE_PATH)     # one bulk copy, not per-record

if MEDLINE_PATH.exists():
    with gzip.open(MEDLINE_PATH, "rt", encoding="utf-8") as fh:
        for line in fh:
            if line.strip():
                row = json.loads(line)
                MEDLINE_STORE[row["pmid"]] = row["medline"]
    print("[prewarm] loaded %d MEDLINE records from Drive" % len(MEDLINE_STORE))

# ---- 1. fetch + parse the shard -------------------------------------------
seed_from_drive()                       # ONE bulk rsync, not 2,000 Drive stats
started = time.time()
_5C_MARK.update(stage="fetch+parse", n=0, total=len(SAMPLE))
PAPER_REFCOUNT, claimed_pmids, xml_failures = {}, set(), []
print("[prewarm] fetching + parsing %d papers with %d workers..."
      % (len(SAMPLE), MAX_WORKERS), flush=True)

def _prepare(pmcid):
    path, source = fetch_citing_xml(pmcid)
    refs = parser.parse_pmc_xml(str(path), source_pmcid=pmcid)
    return pmcid, len(refs), [r.claimed.claimed_pmid.strip() for r in refs
                              if r.claimed.claimed_pmid]

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    futures = {pool.submit(_prepare, p): p for p in SAMPLE}
    for n, future in enumerate(as_completed(futures), 1):
        pmcid = futures[future]
        try:
            pmcid, count, pmids = future.result()
            PAPER_REFCOUNT[pmcid] = count
            claimed_pmids.update(p for p in pmids if PMID_RE.match(p))
        except Exception as exc:
            xml_failures.append({"pmcid": pmcid,
                                 "exception_type": type(exc).__name__,
                                 "message": str(exc)[:300]})
        _5C_MARK["n"] = n
        if n <= 10 or n % 100 == 0 or n == len(SAMPLE):
            print("[prewarm] parsed %d/%d papers | %d unique claimed PMIDs | %.0f s"
                  % (n, len(SAMPLE), len(claimed_pmids), time.time() - started),
                  flush=True)

in_window = [p for p, c in PAPER_REFCOUNT.items()
             if MIN_REFERENCES <= c <= MAX_REFERENCES]
print("[prewarm] papers in the reference-count window: %d of %d | XML failures: %d"
      % (len(in_window), len(PAPER_REFCOUNT), len(xml_failures)))

# ---- 2. batched MEDLINE prewarm -------------------------------------------
def _split_medline(text):
    out = {}
    for chunk in re.split(r"(?m)^(?=PMID-\s*\d)", text):
        m = re.match(r"PMID-\s*(\d+)", chunk)
        if m:
            out[m.group(1)] = chunk
    return out

if PREWARM_PUBMED:
    todo = sorted(p for p in claimed_pmids if p not in MEDLINE_STORE)
    _5C_MARK.update(stage="medline", n=0, total=len(todo))
    print("[prewarm] MEDLINE: %d to fetch in %d batches of %d"
          % (len(todo), -(-len(todo) // EFETCH_BATCH), EFETCH_BATCH), flush=True)
    t0, refused = time.time(), 0
    for start in range(0, len(todo), EFETCH_BATCH):
        batch = todo[start:start + EFETCH_BATCH]
        ratelimit.NCBI.wait()
        try:
            r = SESSION.get(EFETCH_URL, params={
                "db": "pubmed", "id": ",".join(batch), "rettype": "medline",
                "retmode": "text", "api_key": NCBI_API_KEY, "email": EMAIL,
                "tool": "CitationRepairEngine"}, timeout=180)
        except Exception:
            refused += len(batch)
            continue
        if r.status_code != 200:
            # CACHE NOTHING. A non-200 is an outage, and an outage recorded as ""
            # would become manufactured fabrication evidence.
            refused += len(batch)
            continue
        found = _split_medline(r.text)
        with _STORE_LOCK:
            for pmid in batch:
                MEDLINE_STORE[pmid] = found.get(pmid, "")
        done = min(start + EFETCH_BATCH, len(todo))
        _5C_MARK["n"] = done
        if start == 0 or done % 2000 < EFETCH_BATCH or done == len(todo):
            print("[prewarm] MEDLINE %d/%d | %.0f s"
                  % (done, len(todo), time.time() - t0), flush=True)
    absent = sum(1 for v in MEDLINE_STORE.values() if v == "")
    print("[prewarm] MEDLINE store: %d records (%d verified absent), %d left live"
          % (len(MEDLINE_STORE), absent, refused))
    tmp = MEDLINE_PATH.with_suffix(".tmp.gz")
    with gzip.open(tmp, "wt", encoding="utf-8") as fh:
        for pmid, chunk in MEDLINE_STORE.items():
            fh.write(json.dumps({"pmid": pmid, "medline": chunk}) + "\n")
    os.replace(tmp, MEDLINE_PATH)
    shutil.copy2(MEDLINE_PATH, MEDLINE_DRIVE)     # one file, one Drive write

# ---- 3. batched F8 metadata prewarm ---------------------------------------
if PREWARM_F8:
    todo = sorted(p for p in claimed_pmids if p not in F8_STORE)
    _5C_MARK.update(stage="f8", n=0, total=len(todo))
    print("[prewarm] F8 metadata: %d to fetch in %d batches"
          % (len(todo), -(-len(todo) // 200)), flush=True)
    t0 = time.time()
    for start in range(0, len(todo), 200):
        batch = todo[start:start + 200]
        try:
            records, missing = _F8_FINDER._fetch_metadata(batch)
        except Exception:
            continue                      # falls through to live per-pmid
        with _STORE_LOCK:
            for pmid in batch:
                if pmid in records:
                    F8_STORE[pmid] = records[pmid]
                elif pmid in missing:
                    F8_STORE[pmid] = None
        done = min(start + 200, len(todo))
        _5C_MARK["n"] = done
        if start == 0 or done % 2000 < 200 or done == len(todo):
            print("[prewarm] F8 %d/%d | %.0f s"
                  % (done, len(todo), time.time() - t0), flush=True)
    print("[prewarm] F8 store: %d records" % len(F8_STORE))

# ---- 4. VERIFY, then install ----------------------------------------------
_5C_MARK.update(stage="verify", n=0, total=PREWARM_VERIFY_N)
if PREWARM_PUBMED:
    absents = [p for p in MEDLINE_STORE if MEDLINE_STORE[p] == ""]
    present = [p for p in MEDLINE_STORE if MEDLINE_STORE[p] != ""]
    vr = random.Random(SAMPLE_SEED + 7)
    half = PREWARM_VERIFY_N // 2
    n_absent = min(half, len(absents))
    check = (vr.sample(absents, n_absent)
             + vr.sample(present, min(PREWARM_VERIFY_N - n_absent, len(present))))
    mismatches = []
    for pmid in check:
        live = _ORIG_FETCH_PUBMED(pmid, NCBI_API_KEY, EMAIL, None)
        batched = _record_from_chunk(pmid, MEDLINE_STORE[pmid])
        if asdict(live) != asdict(batched):
            mismatches.append({"pmid": pmid, "live": asdict(live),
                               "batched": asdict(batched)})
    atomic_json(MEASURE_ROOT / "prewarm_verification.json", {
        "schema": "cre_prewarm_verification_v1", "checked": len(check),
        "absent_checked": n_absent, "mismatches": mismatches,
        "store_size": len(MEDLINE_STORE), "engine_commit": HEAD, "ts": utc_now()})
    if mismatches:
        raise RuntimeError(
            "Batched EFetch does not reproduce lookup.fetch_pubmed on %d of %d "
            "checked PMIDs. REFUSING to install the prewarm patch. Set "
            "PREWARM_PUBMED = False and rerun, and read "
            "measurements/prewarm_verification.json."
            % (len(mismatches), len(check)))
    print("[prewarm] VERIFIED %d/%d records reproduce lookup.fetch_pubmed exactly "
          "(%d of them ABSENT records)" % (len(check), len(check), n_absent))
    f1_run.fetch_pubmed = cached_fetch_pubmed
    print("[prewarm] INSTALLED: run.fetch_pubmed reads the store first. This is a "
          "RUNTIME PATCH and is recorded in measurements/prewarm_receipt.json.")

atomic_json(MEASURE_ROOT / "prewarm_receipt.json", {
    "schema": "cre_f1f8_prewarm_receipt_v1", "engine_commit": HEAD,
    "papers_prepared": len(PAPER_REFCOUNT),
    "papers_in_reference_window": len(in_window),
    "xml_failures": xml_failures,
    "unique_claimed_pmids": len(claimed_pmids),
    "medline_store": len(MEDLINE_STORE), "f8_store": len(F8_STORE),
    "pubmed_patch_installed": bool(PREWARM_PUBMED),
    "f1_reachable": bool(F1_REACHABLE),
    "runtime_patch_note": ("run.fetch_pubmed was replaced by a cache-read wrapper "
                           "that delegates to the original on a miss. Records are "
                           "parsed by lookup._parse_medline. Equality with the "
                           "original was verified on a sample that included "
                           "ABSENT records. No other engine symbol was touched."),
    "elapsed_s": round(time.time() - started, 1), "ts": utc_now()})
_5C_STOP.set()
sync_to_drive("prewarm")
print("PREWARM DONE in %.0f s" % (time.time() - started))

## 5D. Pre-flight — one paper end to end through the patched path

In [ ]:
# Estimated runtime: ~20-60 seconds (one paper; the NCBI legs are cache reads now).
PREFLIGHT_DIR = RUN_ROOT / "preflight"
shutil.rmtree(PREFLIGHT_DIR, ignore_errors=True)
PREFLIGHT_DIR.mkdir(parents=True, exist_ok=True)

probe = None
for candidate in SAMPLE:
    try:
        xml_path, source = fetch_citing_xml(candidate)
        refs = parser.parse_pmc_xml(str(xml_path), source_pmcid=candidate)
    except Exception as exc:
        print("[preflight] skip", candidate, type(exc).__name__, str(exc)[:90])
        continue
    if MIN_REFERENCES <= len(refs) <= MAX_REFERENCES:
        probe = (candidate, xml_path, refs, source)
        break
if probe is None:
    raise RuntimeError("no usable pre-flight paper in the sample")

pmcid, xml_path, refs, source = probe
print("[preflight] paper", pmcid, "|", len(refs), "references |", source)

counts = f1_run.run(
    str(xml_path.parent),
    str(PREFLIGHT_DIR / "band1_predictions.jsonl"),
    str(PREFLIGHT_DIR / "band1_lossless_log.jsonl"),
    model=MODEL, anthropic_key=ANTHROPIC_API_KEY, ncbi_key=NCBI_API_KEY,
    crossref_mailto=EMAIL, openalex_mailto=EMAIL,
    openalex_api_key=OPENALEX_API_KEY,
    refs=refs, complete=BAND1_COMPLETE,
    f8_timing=True, f8_fetch_meta=f8_fetch_meta, f8_resolve_doi=f8_resolve_doi)

log_rows = read_jsonl(PREFLIGHT_DIR / "band1_lossless_log.jsonl")
f8_status = collections.Counter(
    str((r.get("log") or {}).get("f8_timing_status") or "") for r in log_rows)

print("[preflight] label counts :", dict(counts))
print("[preflight] F8 statuses  :", dict(f8_status))
print("[preflight] F1 rows      :", sum(1 for r in log_rows if r.get("label") == "F1"))
print("[preflight] F8 rows      :", sum(1 for r in log_rows if r.get("label") == "F8"))
print("[preflight] spend so far : $%.4f over %d model calls"
      % (SPEND.usd(), SPEND.calls))
assert log_rows, "Band 1 wrote an empty lossless log"
assert any((r.get("log") or {}).get("f8_timing_version") for r in log_rows), \
    "f8_timing did not run; F8 would be silently absent"
print("PRE-FLIGHT OK — F1/F2/F8 pass executed, F8 timing wired, F3-F7 not called")

## 6. The run — Band 1 only, checkpointed, with live F1/F8 counts

Set `ENABLE_PAID_RUN = True` in Section 2 and rerun that cell first.

**What it does per paper:** fetch the citing XML (cached), parse it, apply the
reference-count guard, run the real Band-1 pass, write
`band1_predictions.jsonl` + `band1_lossless_log.jsonl` + `status.json` into that
paper's own directory on Drive. A paper whose `status.json` says `complete` is
skipped, so this cell is safe to rerun after a disconnect.

**What prints:** every F1 and every F8 the moment it is decided, plus a heartbeat
every 30 seconds carrying the running **F1 and F8 totals**, papers done,
references processed, spend, and rate.

Interrupting is safe. Finished papers stay finished.

**Speed, in order of size.** Section 5C's batched prewarm is the big one — it turns
two NCBI requests per reference into ~400 requests total. After that, sharding:
set `SHARD_COUNT = 4`, pin the same `RUN_NAME` in all four, and run this notebook in
four Colab runtimes with `SHARD_INDEX = 0,1,2,3`. Each runtime is a separate IP with
its own provider budgets, so wall clock falls roughly linearly. A second NCBI API key
in *this* runtime buys nothing — NCBI's limit is per site, not per key.

In [ ]:
# Estimated runtime: ~5-10 minutes per 100-paper batch after Section 5C. Rough —
# a request-count model, not a measurement; the per-paper lines print a real
# p/min within the first minute, so resize from those. ONE BATCH PER EXECUTION:
# rerun this cell for the next 100, up to the SAMPLE_SIZE cap. Checkpointed, so
# a disconnect costs at most one paper.
if not ENABLE_PAID_RUN:
    raise RuntimeError("ENABLE_PAID_RUN is False. Read Section 6, set it True in "
                       "Section 2, rerun that cell, then rerun this one.")

HEARTBEAT_SECONDS = 10          # was 30; this cell is meant to be noisy

# ---- OPENALEX BUDGET GUARD -------------------------------------------------
# openalex_telemetry defines the spent-allowance code as "409", but OpenAlex
# signals insufficient budget with 429 and an "Insufficient budget" body.
# Measured 2026-08-24: 786 calls, 786 x 429, quota_exhausted == 0. Without this,
# an overnight run goes F1-blind and says nothing until morning.
OA_DEAD = threading.Event()
OA_429_TRIP = 25                # refusals before we call the budget spent
OA_WATCH_SECONDS = 30

def openalex_budget(verbose=True):
    """(has_budget, detail). Reads the real body, not just the status code."""
    try:
        r = requests.get("https://api.openalex.org/works",
                         params={"filter": "title.search:hallmarks of cancer",
                                 "per-page": 1, "mailto": EMAIL,
                                 "api_key": OPENALEX_API_KEY}, timeout=20)
    except Exception as exc:
        return None, {"error": "%s: %s" % (type(exc).__name__, exc)}
    if r.status_code == 200:
        if verbose:
            print("OpenAlex: HTTP 200 - budget available", flush=True)
        return True, {"status": 200}
    try:
        body = r.json()
    except Exception:
        body = {"text": r.text[:300]}
    spent = (r.status_code == 429
             and "insufficient budget" in str(body.get("message", "")).lower())
    if verbose:
        print("OpenAlex: HTTP %d | daily $%s | prepaid $%s | retryAfter %s s"
              % (r.status_code, body.get("dailyRemainingUsd"),
                 body.get("prepaidRemainingUsd"), body.get("retryAfter")),
              flush=True)
    return (False if spent else None), {"status": r.status_code, **body}

def openalex_429s(before):
    """Total 429s across EVERY leg. This is what quota_exhausted should report
    and does not. Counting all legs matters: doi and candidates were 46% and 38%
    of demand on 2026-08-24, so they hit the wall before confirm does."""
    legs = openalex_telemetry.delta(before)["legs"]
    return sum(v.get("429", 0) for v in legs.values())

def openalex_watchdog(stop_event, before):
    while not stop_event.wait(OA_WATCH_SECONDS):
        n = openalex_429s(before)
        if n >= OA_429_TRIP and not OA_DEAD.is_set():
            OA_DEAD.set()
            print("\n" + "!" * 74, flush=True)
            print("OPENALEX REFUSED %d CALLS (HTTP 429). Budget is spent." % n,
                  flush=True)
            print("fully_answered() is now False for every reference: F1 is "
                  "BLIND and further papers would be unreportable. Halting new "
                  "papers. Finished papers are unaffected. F8 is unaffected.",
                  flush=True)
            print("!" * 74 + "\n", flush=True)
            return

STATE = {
    "papers_done": 0, "papers_skipped": 0, "papers_failed": 0,
    "references": 0, "F1": 0, "F8": 0, "F2_computed_excluded": 0,
    "quarantined": 0,
}
STATE_LOCK = threading.Lock()
STOP = threading.Event()
STARTED = time.time()

# ---- one lock for the console, so 24 threads cannot interleave a line -------
PRINT_LOCK = threading.Lock()
def say(msg):
    with PRINT_LOCK:
        print(msg, flush=True)

# ---- in-flight registry, so a STALL is visible instead of silent -----------
INFLIGHT = {}                   # pmcid -> (started_epoch, stage)
INFLIGHT_LOCK = threading.Lock()
def mark(pmcid, stage):
    with INFLIGHT_LOCK:
        if stage is None:
            INFLIGHT.pop(pmcid, None)
        else:
            t0 = INFLIGHT.get(pmcid, (time.time(), ""))[0]
            INFLIGHT[pmcid] = (t0, stage)

def already_complete(pmcid):
    status = paper_dir(pmcid) / "status.json"
    if not status.exists():
        return None
    try:
        payload = json.loads(status.read_text(encoding="utf-8"))
    except Exception:
        return None
    return payload if payload.get("status") in {"complete", "skipped"} else None

def process_paper(pmcid):
    out = paper_dir(pmcid)
    out.mkdir(parents=True, exist_ok=True)
    dataset = out / "band1_predictions.jsonl"
    logpath = out / "band1_lossless_log.jsonl"
    started = time.time()
    mark(pmcid, "fetch")

    xml_path, source = fetch_citing_xml(pmcid)
    mark(pmcid, "parse")
    refs = parser.parse_pmc_xml(str(xml_path), source_pmcid=pmcid)
    if not (MIN_REFERENCES <= len(refs) <= MAX_REFERENCES):
        payload = {"pmcid": pmcid, "status": "skipped", "references": len(refs),
                   "reason": "reference_count_outside_%d_%d"
                             % (MIN_REFERENCES, MAX_REFERENCES), "ts": utc_now()}
        atomic_json(out / "status.json", payload)
        mark(pmcid, None)
        return payload

    for stale in (dataset, logpath):
        if stale.exists():
            stale.unlink()

    mark(pmcid, "band1(%d refs)" % len(refs))
    buf = io.StringIO()
    STDOUT_ROUTER.park(buf)
    try:
        counts = f1_run.run(
            str(xml_path.parent), str(dataset), str(logpath),
            model=MODEL, anthropic_key=ANTHROPIC_API_KEY, ncbi_key=NCBI_API_KEY,
            crossref_mailto=EMAIL, openalex_mailto=EMAIL,
            openalex_api_key=OPENALEX_API_KEY,
            refs=refs, complete=BAND1_COMPLETE,
            f8_timing=True, f8_fetch_meta=f8_fetch_meta,
            f8_resolve_doi=f8_resolve_doi)
    finally:
        STDOUT_ROUTER.release()
    (out / "band1_stdout.txt").write_text(buf.getvalue(), encoding="utf-8")

    rows = read_jsonl(logpath)
    hits = [r for r in rows if r.get("label") in ("F1", "F8")]
    for row in hits:
        log = row.get("log") or {}
        append_jsonl(FINDINGS_LIVE, {
            "ts": utc_now(), "citing_pmcid": pmcid,
            "citation_id": row.get("citation_id"), "label": row.get("label"),
            "confidence": row.get("confidence"), "rationale": row.get("rationale"),
            "decided_by": log.get("decided_by"),
            "f8_timing_status": log.get("f8_timing_status"),
            "f8_timing_reason": log.get("f8_timing_reason"),
            "f8_notice_date": log.get("f8_notice_date"),
            "f8_timing_gap_days": log.get("f8_timing_gap_days"),
            "record": row})
        say("  >>> %s  %s  %s" % (row.get("label"), row.get("citation_id"),
                                  str(row.get("rationale") or "")[:100]))

    payload = {
        "pmcid": pmcid, "status": "complete", "references": len(rows),
        "label_counts": {k: int(v) for k, v in counts.items()},
        "f1": sum(1 for r in rows if r.get("label") == "F1"),
        "f8": sum(1 for r in rows if r.get("label") == "F8"),
        "f2_computed_excluded": sum(1 for r in rows if r.get("label") == "F2"),
        "quarantined": sum(1 for r in rows
                           if (r.get("log") or {}).get("decided_by")
                           == "quarantine_exception"),
        "xml_source": source, "elapsed_s": round(time.time() - started, 2),
        "engine_commit": HEAD, "ts": utc_now(),
    }
    atomic_json(out / "status.json", payload)
    mark(pmcid, None)
    return payload

def fold(payload):
    with STATE_LOCK:
        if payload.get("status") == "skipped":
            STATE["papers_skipped"] += 1
            return dict(STATE)
        STATE["papers_done"] += 1
        STATE["references"] += int(payload.get("references") or 0)
        STATE["F1"] += int(payload.get("f1") or 0)
        STATE["F8"] += int(payload.get("f8") or 0)
        STATE["F2_computed_excluded"] += int(payload.get("f2_computed_excluded") or 0)
        STATE["quarantined"] += int(payload.get("quarantined") or 0)
        return dict(STATE)

def progress_line(payload, snap):
    """ONE line per finished paper, with the running F1/F8 totals on it."""
    attempted = snap["papers_done"] + snap["papers_skipped"] + STATE["papers_failed"]
    elapsed = time.time() - STARTED
    ppm = (attempted / elapsed * 60.0) if elapsed > 0 else 0.0
    eta = ((len(TODO) - attempted) / ppm) if ppm > 0 else float("nan")
    if payload.get("status") == "skipped":
        return ("[%4d/%d] %-12s SKIP %d refs (guard)          | F1 %d F8 %d "
                "| %.1f p/min | eta %.0fm"
                % (attempted, len(TODO), payload["pmcid"], payload["references"],
                   snap["F1"], snap["F8"], ppm, eta))
    return ("[%4d/%d] %-12s %3d refs  f1=%d f8=%d  %5.1fs | F1 %d F8 %d "
            "| $%.2f | %.1f p/min | eta %.0fm"
            % (attempted, len(TODO), payload["pmcid"], payload["references"],
               payload.get("f1", 0), payload.get("f8", 0),
               payload.get("elapsed_s", 0.0), snap["F1"], snap["F8"],
               SPEND.usd(), ppm, eta))

def heartbeat():
    """Fires every HEARTBEAT_SECONDS no matter what the workers are doing, so
    silence is never ambiguous. Names the oldest in-flight paper: if that number
    keeps climbing and nothing completes, the run is stalled, not slow."""
    while not STOP.wait(HEARTBEAT_SECONDS):
        with STATE_LOCK:
            s = dict(STATE)
        with INFLIGHT_LOCK:
            flight = sorted(INFLIGHT.items(), key=lambda kv: kv[1][0])
        elapsed = time.time() - STARTED
        attempted = s["papers_done"] + s["papers_skipped"] + s["papers_failed"]
        rate = (s["references"] / elapsed) if elapsed > 0 else 0.0
        eta = ((len(TODO) - attempted) / (attempted / elapsed)) if attempted else float("nan")
        hit, live = PREWARM_STATS["pubmed_hit"], PREWARM_STATS["pubmed_live"]
        hit_pct = (100.0 * hit / (hit + live)) if (hit + live) else 0.0
        # delta() returns {"legs": {...}, "leg_totals": {...}, "total": n,
        # "quota_exhausted": n} -- the spent-allowance count is already hoisted.
        oa409 = openalex_telemetry.delta(OA_BEFORE)["quota_exhausted"]
        oldest = ""
        if flight:
            pmcid, (t0, stage) = flight[0]
            oldest = " | oldest %s %s %.0fs" % (pmcid, stage, time.time() - t0)
        say("[hb %5.1fm] papers %d/%d (skip %d fail %d) | refs %d | "
            "F1 %d | F8 %d | F2excl %d | $%.2f | %.1f refs/s | cache %.0f%% | "
            "OA409 %d | in-flight %d%s | eta %.0fm"
            % (elapsed / 60.0, s["papers_done"], len(TODO), s["papers_skipped"],
               s["papers_failed"], s["references"], s["F1"], s["F8"],
               s["F2_computed_excluded"], SPEND.usd(), rate, hit_pct,
               oa409, len(flight), oldest, eta / 60.0))

# ---- resume ----------------------------------------------------------------
if "seed_from_drive" in globals():
    seed_from_drive()
say("[run] scanning %d papers for finished work..." % len(SAMPLE))
REMAINING = []
for pmcid in SAMPLE:
    done = already_complete(pmcid)
    if done is None:
        REMAINING.append(pmcid)
    else:
        fold(done)

# ONE BATCH PER EXECUTION. Rerun this cell for the next 100; the resume scan
# above is what makes that safe. SAMPLE_SIZE is the hard cap and is never
# exceeded because REMAINING is drawn from SAMPLE.
TODO = REMAINING[:BATCH_SIZE]
say("[run] cap %d | done %d | remaining %d | THIS BATCH %d"
    % (SAMPLE_SIZE, len(SAMPLE) - len(REMAINING), len(REMAINING), len(TODO)))
say("[run] carried in from finished papers: F1 %d, F8 %d"
    % (STATE["F1"], STATE["F8"]))
if not TODO:
    say("[run] NOTHING LEFT. All %d papers in the cap are done. Go to Section 7."
        % SAMPLE_SIZE)
say("[run] STARTING %s with %d workers. A line per paper, a heartbeat every %ds."
    % (utc_now(), MAX_WORKERS, HEARTBEAT_SECONDS))
say("[run] model: %s | prices checked %s" % (MODEL, PRICE_CHECKED_DATE))

_ok, _detail = openalex_budget()
if _ok is False:
    raise RuntimeError(
        "OpenAlex budget is $0 (resets at midnight UTC, or add prepaid usage in "
        "$1 increments at https://openalex.org/pricing). Starting now would make "
        "every F1 in this batch a hold, exactly like the 327-paper Opus batch. "
        "F8 would still be valid -- if that is what you want, comment out this "
        "check deliberately and record it.")

OA_BEFORE = openalex_telemetry.snapshot()
threading.Thread(target=openalex_watchdog, args=(STOP, OA_BEFORE),
                 daemon=True).start()

hb = threading.Thread(target=heartbeat, daemon=True)
hb.start()
try:
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = {pool.submit(process_paper, p): p for p in TODO}
        say("[run] %d papers submitted; waiting on the first completion..."
            % len(futures))
        for future in as_completed(futures):
            pmcid = futures[future]
            try:
                payload = future.result()
                snap = fold(payload)
                say(progress_line(payload, snap))
                if "sync_to_drive" in globals():
                    sync_to_drive("papers", min_interval=300)
                if OA_DEAD.is_set():
                    for other in futures:
                        other.cancel()
                    say("[halt] OpenAlex budget spent - cancelled the remaining "
                        "papers. Finished ones are on disk.")
                    break
            except f1_run.NonRetryableProviderError as exc:
                say("[FATAL] %s: %s" % (type(exc).__name__, exc))
                for other in futures:
                    other.cancel()
                raise
            except Exception as exc:
                with STATE_LOCK:
                    STATE["papers_failed"] += 1
                append_jsonl(FAILURES, {"ts": utc_now(), "pmcid": pmcid,
                                        "exception_type": type(exc).__name__,
                                        "message": str(exc)[:500]})
                say("[fail] %s %s: %s" % (pmcid, type(exc).__name__,
                                          str(exc)[:110]))
            finally:
                mark(pmcid, None)
finally:
    STOP.set()
    hb.join(timeout=2)
    if "sync_to_drive" in globals():
        sync_to_drive("run end")

# ---- THE EXIT GATE ---------------------------------------------------------
# OpenAlex answered at the START of the 2026-08-24 run and was dead by the end,
# which is how an artifact F1=0 got produced over 8,009 references. A probe
# before the batch proves nothing about the batch. Probe again, now.
OA_DELTA = openalex_telemetry.delta(OA_BEFORE)
# 429, not the telemetry's "quota_exhausted" (which keys on 409 and reads 0
# while every call is being budget-refused).
OA_429 = sum(v.get("429", 0) for v in OA_DELTA["legs"].values())
OA_409 = OA_DELTA["quota_exhausted"]
exit_hits = {
    "pubmed":   f1_confirm.search_pubmed(PROBE_TITLE, NCBI_API_KEY),
    "crossref": f1_confirm.search_crossref(PROBE_TITLE, EMAIL),
    "openalex": f1_confirm.search_openalex(PROBE_TITLE, EMAIL,
                                           api_key=OPENALEX_API_KEY),
}
F1_REACHABLE_AFTER = f1_confirm.fully_answered(exit_hits)
CONFIRM_409 = OA_DELTA["legs"].get("confirm", {}).get("409", 0)
CONFIRM_429 = OA_DELTA["legs"].get("confirm", {}).get("429", 0)
F1_REPORTABLE = bool(F1_REACHABLE and F1_REACHABLE_AFTER
                     and CONFIRM_409 == 0 and CONFIRM_429 == 0)
atomic_json(MEASURE_ROOT / "provider_probe_after.json", {
    "schema": "cre_provider_probe_v1", "when": "after_batch",
    "probe_title": PROBE_TITLE, "hits": exit_hits,
    "f1_reachable": F1_REACHABLE_AFTER,
    "f1_reachable_before": bool(F1_REACHABLE),
    "f1_reportable_this_batch": F1_REPORTABLE,
    "openalex_calls_by_leg": OA_DELTA, "confirm_leg_409": CONFIRM_409,
    "confirm_leg_429": CONFIRM_429, "openalex_429_total": OA_429,
    "model": MODEL, "halted_on_openalex": OA_DEAD.is_set(),
    "engine_commit": HEAD, "ts": utc_now()})

atomic_json(MEASURE_ROOT / "run_state.json",
            {**STATE, "sample_size": len(SAMPLE), "queued": len(TODO),
             "elapsed_s": round(time.time() - STARTED, 1),
             "model_calls": SPEND.calls, "spend_usd": round(SPEND.usd(), 4),
             "prewarm_stats": dict(PREWARM_STATS),
             "batch_size": BATCH_SIZE, "cap": SAMPLE_SIZE,
             "f1_reachable_before": bool(F1_REACHABLE),
             "f1_reachable_after": F1_REACHABLE_AFTER,
             "f1_reportable_this_batch": F1_REPORTABLE,
             "openalex_calls_by_leg": OA_DELTA,
             "openalex_authenticated": bool(OPENALEX_API_KEY),
             "openalex_429_total": OA_429, "halted_on_openalex": OA_DEAD.is_set(),
             "model": MODEL, "engine_commit": HEAD, "finished_at": utc_now()})

print("\nRUN COMPLETE")
print("papers done      :", STATE["papers_done"])
print("papers skipped   :", STATE["papers_skipped"], "(reference count guard)")
print("papers failed    :", STATE["papers_failed"])
print("references       :", STATE["references"])
print("F1 findings      :", STATE["F1"])
print("F8 findings      :", STATE["F8"])
print("F2 computed, EXCLUDED from the reported set:", STATE["F2_computed_excluded"])
print("spend            : $%.2f over %d model calls" % (SPEND.usd(), SPEND.calls))
print("prewarm cache    :", dict(PREWARM_STATS))
print("openalex by leg  :", OA_DELTA["leg_totals"],
      "| total", OA_DELTA["total"], "| quota_exhausted", OA_409)
print("\nF1 REPORTABILITY GATE")
print("  reachable BEFORE batch :", bool(F1_REACHABLE))
print("  reachable AFTER  batch :", F1_REACHABLE_AFTER, exit_hits)
print("  confirm-leg 409s / 429s:", CONFIRM_409, "/", CONFIRM_429)
print("  openalex 429s, all legs:", OA_429)
print("  halted on openalex     :", OA_DEAD.is_set())
if F1_REPORTABLE:
    print("  --> F1 IS REPORTABLE for this batch. The F1 count above is a finding.")
else:
    print("  --> F1 IS NOT REPORTABLE for this batch. OpenAlex stopped answering, "
          "so confirm.fully_answered() was False for some or all references and "
          "F1 was held rather than decided. Record F1 as NOT ATTEMPTED for these "
          "papers, never as zero. F8 is unaffected: it never calls confirm().")
say("\n[run] batch done. Rerun this cell for the next %d, or go to Section 7."
    % BATCH_SIZE)

## 7. Aggregate — canonical disposition and the F1/F8 funnel

The disposition is built by the engine's own `preband_disposition.write_disposition`
over every paper's lossless log, so the population every rate below is a fraction of
is the canonical artifact and not something this notebook counted for itself.

In [ ]:
# Estimated runtime: ~1-5 minutes for 2,000 papers (reads every per-paper log).
ALL_LOG = FAST_RUN / "band1_lossless_log_all.jsonl"
ALL_PRED = FAST_RUN / "band1_predictions_all.jsonl"

# Aggregates the FULL sample, so running this in any one shard rolls up every
# shard's papers -- they all write under the same RUN_NAME on Drive.
completed, skipped = [], []
with ALL_LOG.open("w", encoding="utf-8") as log_out, \
     ALL_PRED.open("w", encoding="utf-8") as pred_out:
    for pmcid in FULL_SAMPLE:
        status_path = paper_dir(pmcid) / "status.json"
        if not status_path.exists():
            continue
        payload = json.loads(status_path.read_text(encoding="utf-8"))
        if payload.get("status") != "complete":
            skipped.append(payload)
            continue
        completed.append(payload)
        for name, handle in (("band1_lossless_log.jsonl", log_out),
                             ("band1_predictions.jsonl", pred_out)):
            path = paper_dir(pmcid) / name
            if path.exists():
                with path.open(encoding="utf-8") as fh:
                    shutil.copyfileobj(fh, handle)

DISPOSITION = FAST_RUN / "preband_disposition_v1.jsonl"
disposition_manifest = preband_disposition.write_disposition(
    str(ALL_LOG), str(DISPOSITION), f2_commit=HEAD,
    generated_by="F1F8_ONLY notebook (Band 1 only; F3-F7 not executed)",
    generated_at=SNAPSHOT_DATE)

rows = read_jsonl(ALL_LOG)
labels = collections.Counter(str(r.get("label") or "") for r in rows)
f8_status = collections.Counter(
    str((r.get("log") or {}).get("f8_timing_status") or "(none)") for r in rows)
f8_reason = collections.Counter(
    str((r.get("log") or {}).get("f8_timing_reason") or "(none)") for r in rows)
f1_decided_by = collections.Counter(
    str((r.get("log") or {}).get("decided_by") or "") for r in rows
    if r.get("label") == "F1")
f8_decided_by = collections.Counter(
    str((r.get("log") or {}).get("decided_by") or "") for r in rows
    if r.get("label") == "F8")

FUNNEL = {
    "schema": "cre_f1f8_only_funnel_v1",
    "engine_commit": HEAD,
    "bands_executed": ["band1"],
    "taxonomies_executed": ["F1", "F2", "F8"],
    "taxonomies_reported": ["F1", "F8"],
    "taxonomies_not_executed": ["F3", "F4", "F5", "F6", "F7"],
    "papers_sampled": len(FULL_SAMPLE),
    "shard_count": SHARD_COUNT,
    "papers_complete": len(completed),
    "papers_skipped_reference_guard": len(skipped),
    "papers_failed": len(read_jsonl(FAILURES)),
    "references_processed": len(rows),
    "label_counts_all": dict(sorted(labels.items())),
    "disposition_label_counts": disposition_manifest["label_counts"],
    "f1_reportable_last_batch": bool(globals().get("F1_REPORTABLE", False)),
    "F1_findings": labels.get("F1", 0),
    "F8_findings": labels.get("F8", 0),
    "F2_computed_excluded": labels.get("F2", 0),
    "F1_decided_by": dict(f1_decided_by),
    "F8_decided_by": dict(f8_decided_by),
    "f8_timing_status_counts": dict(sorted(f8_status.items())),
    "f8_timing_reason_counts": dict(sorted(f8_reason.items())),
    "quarantined": sum(1 for r in rows
                       if (r.get("log") or {}).get("decided_by")
                       == "quarantine_exception"),
    "disposition_path": str(DISPOSITION),
    "disposition_artifact_sha256": disposition_manifest["artifact_sha256"],
    "spend_usd": round(SPEND.usd(), 4), "model_calls": SPEND.calls,
    "generated_at": utc_now(),
}
atomic_json(MEASURE_ROOT / "f1_f8_funnel.json", FUNNEL)
sync_to_drive("aggregate")

print("references processed :", FUNNEL["references_processed"])
print("F1 findings          :", FUNNEL["F1_findings"])
print("F8 findings          :", FUNNEL["F8_findings"])
print("F2 computed, excluded:", FUNNEL["F2_computed_excluded"])
print("quarantined (unjudged, never a finding):", FUNNEL["quarantined"])
print("\nF8 timing status:")
for k, v in sorted(f8_status.items(), key=lambda kv: -kv[1]):
    print("  %-24s %d" % (k, v))
print("\nF1 decided_by:", dict(f1_decided_by))
print("F8 decided_by:", dict(f8_decided_by))

## 8. The blind grading queue — the only thing precision can be computed from

Precision is confirmed findings over graded findings. The engine never grades
itself, so this cell emits one row per F1 and F8 finding with **empty**
`human_label` and `human_note` fields, in shuffled order, with the machine's own
label withheld from the reviewer-facing columns.

Fill `human_label` with `correct` or `incorrect`, then rerun the last block for the
count. Anything before that is a candidate count, not a precision figure.

In [ ]:
# Estimated runtime: under 1 minute.
QUEUE = REVIEW_ROOT / "f1_f8_grading_queue.jsonl"
KEY   = REVIEW_ROOT / "f1_f8_grading_key.jsonl"       # machine labels, held back

findings = [r for r in rows if r.get("label") in ("F1", "F8")]
order = random.Random(SAMPLE_SEED + 1)
order.shuffle(findings)

with QUEUE.open("w", encoding="utf-8") as q, KEY.open("w", encoding="utf-8") as k:
    for i, row in enumerate(findings, 1):
        log = row.get("log") or {}
        claimed = row.get("claimed") or {}
        retrieved = row.get("retrieved") or {}
        cid = row.get("citation_id") or ""
        pmcid = cid.split(":", 1)[0]
        q.write(json.dumps({
            "item": i,
            "citation_id": cid,
            "citing_paper": "https://www.ncbi.nlm.nih.gov/pmc/articles/%s/" % pmcid,
            "citance": row.get("citance") or "",
            "claimed_raw": claimed.get("raw") or "",
            "claimed_title": claimed.get("title") or "",
            "claimed_authors": claimed.get("authors") or [],
            "claimed_year": claimed.get("year"),
            "claimed_journal": claimed.get("journal") or "",
            "claimed_pmid": claimed.get("claimed_pmid") or "",
            "claimed_doi": claimed.get("claimed_doi") or "",
            "retrieved_title": retrieved.get("title") or "",
            "retrieved_pmid": retrieved.get("pmid") or "",
            "retrieved_year": retrieved.get("year"),
            "retrieved_resolved": retrieved.get("resolved"),
            "rationale": row.get("rationale") or "",
            "f8_timing_status": log.get("f8_timing_status") or "",
            "f8_notice_date": log.get("f8_notice_date") or "",
            "f8_citing_date_earliest": log.get("f8_citing_date_earliest") or "",
            "f8_timing_gap_days": log.get("f8_timing_gap_days"),
            "human_label": "",     # "correct" | "incorrect"
            "human_note": "",
        }, ensure_ascii=False, sort_keys=True) + "\n")
        k.write(json.dumps({"item": i, "citation_id": cid,
                            "machine_label": row.get("label"),
                            "confidence": row.get("confidence"),
                            "decided_by": log.get("decided_by")},
                           ensure_ascii=False, sort_keys=True) + "\n")

print("grading queue :", QUEUE, "(%d rows)" % len(findings))
print("held-back key :", KEY)
sync_to_drive("grading queue")
print("F1 rows:", sum(1 for r in findings if r["label"] == "F1"),
      "| F8 rows:", sum(1 for r in findings if r["label"] == "F8"))
if not findings:
    print("\nZERO F1 AND ZERO F8 IN THIS SAMPLE. That is a result about the base "
          "rate, not a precision estimate. Precision is undefined on an empty "
          "numerator; report the denominator (%d references) and stop."
          % FUNNEL["references_processed"])

In [ ]:
# Estimated runtime: under 10 seconds. Rerun this AFTER filling human_label.
graded = [r for r in read_jsonl(QUEUE) if str(r.get("human_label") or "").strip()]
key = {r["item"]: r for r in read_jsonl(KEY)}
by_label = collections.defaultdict(lambda: {"graded": 0, "correct": 0})
for row in graded:
    label = key.get(row["item"], {}).get("machine_label", "?")
    by_label[label]["graded"] += 1
    by_label[label]["correct"] += (
        str(row["human_label"]).strip().lower() == "correct")

print("PRECISION (human-graded only; the engine never grades itself)")
if not graded:
    print("  no rows graded yet — fill human_label in", QUEUE)
for label in sorted(by_label):
    d = by_label[label]
    p = d["correct"] / d["graded"] if d["graded"] else float("nan")
    print("  %-3s  %d/%d = %.4f" % (label, d["correct"], d["graded"], p))
atomic_json(MEASURE_ROOT / "f1_f8_precision.json", {
    "schema": "cre_f1f8_precision_v1", "engine_commit": HEAD,
    "graded_rows": len(graded), "by_label": {k: dict(v) for k, v in by_label.items()},
    "denominator_note": ("Precision over HUMAN-GRADED findings only. Not recall. "
                         "F2 was computed by the same Band-1 pass and excluded. "
                         "F3-F7 were never executed."),
    "generated_at": utc_now()})

## What this run can and cannot claim

**Can:** F1 and F8 precision on a uniform random sample of PMC open-access papers
published 2024–2025, at engine commit `5602a3b4`, once the grading queue carries
human labels.

**Cannot:**

- **Recall.** Nothing here searches for missed faults.
- **"F2 was not run."** It was. Band 1 decides F1, F2 and F8 in one shared pass;
  F2 is in the raw artifacts and excluded from the reported set.
- **Anything about F3–F7.** They never executed.
- **A precision figure before human grading.** The counts in Section 7 are
  candidate counts.
- **A population prevalence.** The frame is PMC OA 2024–2025 with the
  `truncated_days` caveat in the frame manifest, not all of the literature.

Everything the run produces lives under `MyDrive/CitationRepairEngine/runs/<RUN_NAME>/`.